# RHEED hardware provenance and RTL simulation
Records the exact source/firmware inventory, simulator version, testbench, commands, results and waveform. The executed stage is **NMS RTL simulation**, not a camera acquisition or a new Vivado build. Use `run_hardware.py` to capture those stages on the lab machine. See DATAERAI.md.

In [ ]:
from pathlib import Path
import os
os.environ["RHEED_DATA_MODE"] = "simulation"
from rheed_runtime import setup
from hardware_runtime import source_manifest, run_tool
dataerai = setup("Hardware_Provenance.ipynb")

In [ ]:
%%dataerai
import json, shutil
root = Path.cwd()
source_inventory = source_manifest(root)
hardware_context = {"stage": "nms_rtl_simulation", "fpga_part": "xcku035-fbva676-2-e", "grid": [40, 40], "top_n": 5, "min_distance_squared": 64, "physical_acquisition": "not_run_no_attached_camera", "synthesis": "not_run_requires_vivado_and_euresys_ip", "implementation": "not_run_requires_vivado", "tools": {name: shutil.which(name) for name in ["iverilog", "vvp", "vivado", "vitis_hls"]}}
print(hardware_context)

In [ ]:
%%dataerai
for source in ["08_hls_design/TopCrop.sv", "tests/tb_dataerai_nms.sv", "hardware_runtime.py", "04_ref_design/CustomLogic.vhd", "04_ref_design/frame_to_line.vhd", "03_scripts/create_vivado_project.tcl", "03_scripts/configure_fgrabber.js"]:
    dataerai.capture_file(root / source, role="source", relationship="uses_dependency")
# This is an existing release, explicitly not rebuilt by this run.
firmware_asset = dataerai.capture_file(root / "06_release/CoaxlinkOcto_1cam.bit", role="firmware", relationship="uses_dependency", metadata={"origin":"existing_repository_release", "rebuilt_this_run":False})

In [ ]:
%%dataerai
simulator_version = run_tool(["iverilog", "-V"], root)
print(simulator_version)
assert simulator_version["status"] == "succeeded", simulator_version

In [ ]:
%%dataerai
work = dataerai.run_dir / "rtl-simulation"
work.mkdir()
compile_result = run_tool(["iverilog", "-g2012", "-s", "tb_dataerai_nms", "-o", str(work / "nms.vvp"), str(root / "08_hls_design/TopCrop.sv"), str(root / "tests/tb_dataerai_nms.sv")], work)
print(compile_result)
assert compile_result["status"] == "succeeded", compile_result

In [ ]:
%%dataerai
simulation_result = run_tool(["vvp", str(work / "nms.vvp")], work)
print(simulation_result)
assert simulation_result["status"] == "succeeded", simulation_result
assert "PASS: NMS" in simulation_result["stdout"]
waveform_asset = dataerai.capture_file(work / "nms-waveform.vcd", role="analysis")

In [ ]:
dataerai.finish()